# Lab | List, Dict and Set Comprehension

## Exercise: Managing Customer Orders Optimized with Comprehension

In the previous exercise, you developed a program to manage customer orders and inventory. Now, let's take it a step further and incorporate comprehension into your code.

Follow the steps below to complete the exercise:

1. Review your code from the previous exercise and identify areas where you can apply comprehension to simplify and streamline your code. 

    - *Hint: Apply it to initialize inventory, updating the inventory and printing the updated inventory.*
    
    - For example, in initializing the inventory, we could have:
    
        ```python
        def initialize_inventory(products):
            inventory = {product: int(input(f"Enter the quantity of {product}s available: ")) for product in products}
            return inventory

        ```
<br>
    
    
2. Modify the function get_customer_orders so it prompts the user to enter the number of customer orders and gathers the product names using a loop and user input. Use comprehension.

3. Add a new function to calculate the total price of the customer order. For each product in customer_orders, prompt the user to enter the price of that product. Use comprehension to calculate the total price. Note: assume that the user can only have 1 unit of each product.

4. Modify the update_inventory function to remove the product from the inventory if its quantity becomes zero after fulfilling the customer orders. Use comprehension to filter out the products with a quantity of zero from the inventory.

5. Print the total price of the customer order.

Your code should produce output similar to the following:

```python
Enter the quantity of t-shirts available:  5
Enter the quantity of mugs available:  4
Enter the quantity of hats available:  3
Enter the quantity of books available:  2
Enter the quantity of keychains available:  1
Enter the number of customer orders:  2
Enter the name of a product that a customer wants to order:  hat
Enter the name of a product that a customer wants to order:  keychain

Order Statistics:
Total Products Ordered: 2
Percentage of Unique Products Ordered: 40.0

Updated Inventory:
t-shirt: 5
mug: 4
hat: 2
book: 2
Enter the price of keychain:  5
Enter the price of hat:  10
Total Price: 15.0

```


## Resolución

Parto del código del lab de *functions* (las mismas funciones y los mismos nombres) y aplico comprensiones donde el enunciado lo pide. En cada paso explico qué hace el código, por qué lo hago así y qué alternativa descarté.

### Paso 0. Lista de productos

Es la misma lista de los labs anteriores. La dejo en una celda aparte porque todas las funciones la usan.

In [1]:
products = ["t-shirt", "mug", "hat", "book", "keychain"]

### Paso 1. `initialize_inventory` con comprensión de diccionario

**Qué hace:** recorre `products` y, para cada producto, pide la cantidad y la guarda como `producto: cantidad`.

**Por qué así:** en el lab de functions teníamos un diccionario vacío, un `for` y una asignación por vuelta. La comprensión `{clave: valor for elemento in lista}` hace lo mismo en una línea y deja claro que el resultado es un diccionario construido a partir de la lista.

**Por qué no más compacta todavía:** se podría devolver directamente la comprensión (`return {...}`), pero guardarla primero en `inventory` hace el código más fácil de leer y de depurar.

In [2]:
def initialize_inventory(products):
    inventory = {product: int(input(f"Enter the quantity of {product}s available: ")) for product in products}
    return inventory

### Paso 2. `get_customer_orders` con número de pedidos y comprensión de conjunto

**Qué hace:** primero pregunta cuántos pedidos hay y luego pide ese número de productos. El resultado es un `set`.

**Por qué así:**
- Antes preguntábamos "¿quieres añadir otro producto?" en cada vuelta. Ahora el enunciado pide preguntar el número al principio, así que sabemos cuántas vueltas dar y podemos usar `range(number_of_orders)`.
- La comprensión de conjunto `{... for _ in range(n)}` sustituye al `set()` vacío más `add()`. Usamos `_` como variable porque el número de la vuelta no nos interesa, solo repetir.
- `.strip().lower()` limpia espacios y mayúsculas, para que `" Hat"` cuente como `"hat"`.

**Por qué un `set` y no una lista:** es lo que devolvía la función en el lab anterior y evita contar dos veces el mismo producto. El precio del Paso 3 asume 1 unidad de cada producto, así que encaja.

In [3]:
def get_customer_orders():
    number_of_orders = int(input("Enter the number of customer orders: "))
    customer_orders = {input("Enter the name of a product that a customer wants to order: ").strip().lower()
                       for _ in range(number_of_orders)}
    return customer_orders

### Paso 3. `update_inventory` con comprensión y filtro de productos agotados

**Qué hace:** en un primer paso resta 1 a cada producto pedido. En un segundo paso quita del inventario los productos que se han quedado a 0.

**Por qué así:**
- La primera comprensión recorre `inventory.items()` y usa una expresión condicional (`cantidad - 1 if producto in pedidos else cantidad`) para decidir el nuevo valor.
- La segunda comprensión usa un `if` al final, que funciona como filtro: solo entran los productos con `quantity > 0`. Es exactamente lo que pide el punto 4.

**Cambio importante respecto al lab de functions:** antes la función modificaba el diccionario "por dentro" (`inventory[product] -= 1`) y no devolvía nada. Una comprensión crea un diccionario **nuevo**, así que ahora la función hace `return` y en el programa principal hay que guardar el resultado: `inventory = update_inventory(...)`.

**Por qué no borrar con `del` dentro de un `for`:** borrar claves de un diccionario mientras lo recorres da error (`RuntimeError: dictionary changed size during iteration`). Filtrar con una comprensión evita ese problema.

In [4]:
def update_inventory(customer_orders, inventory):
    inventory = {product: (quantity - 1 if product in customer_orders else quantity)
                 for product, quantity in inventory.items()}
    inventory = {product: quantity for product, quantity in inventory.items() if quantity > 0}
    return inventory

### Paso 4. Estadísticas del pedido

**Qué hace:** cuenta los productos pedidos y calcula el porcentaje respecto al total de productos de la tienda.

**Por qué igual que antes:** aquí no hay ningún bucle que simplificar, son dos cálculos directos. Meter una comprensión solo por usarla empeoraría la lectura.

In [5]:
def calculate_order_statistics(customer_orders, products):
    total_products_ordered = len(customer_orders)
    percentage_ordered = total_products_ordered / len(products) * 100
    return total_products_ordered, percentage_ordered


def print_order_statistics(order_statistics):
    total_products_ordered, percentage_ordered = order_statistics
    print("\nOrder Statistics:")
    print(f"Total Products Ordered: {total_products_ordered}")
    print(f"Percentage of Unique Products Ordered: {percentage_ordered}")

### Paso 5. Imprimir el inventario actualizado

**Qué hace:** construye todas las líneas `producto: cantidad` con una expresión generadora y las une con saltos de línea en un único `print`.

**Por qué así:** `"\n".join(...)` recibe una comprensión (sin corchetes, es un generador) y produce el texto completo de una vez.

**Por qué no `[print(...) for ...]`:** funciona, pero crea una lista de `None` que no se usa para nada. Las comprensiones sirven para **construir** colecciones, no para ejecutar acciones; para eso está el `for` normal o, como aquí, construir el texto y luego imprimirlo.

In [6]:
def print_updated_inventory(inventory):
    print("\nUpdated Inventory:")
    print("\n".join(f"{product}: {quantity}" for product, quantity in inventory.items()))

### Paso 6. Nueva función `calculate_total_price`

**Qué hace:** para cada producto del pedido pide el precio y suma todos los precios.

**Por qué así:** `sum()` acepta directamente una expresión generadora, así que `sum(float(input(...)) for product in customer_orders)` pide y suma en una sola línea. Convierto a `float` porque un precio puede tener decimales (por eso el resultado sale `15.0`, como en el ejemplo del enunciado).

**Por qué no guardar primero los precios en una lista:** no los necesitamos después. Si hiciera falta un ticket con el precio de cada producto, usaría una comprensión de diccionario `{product: float(input(...)) for product in customer_orders}` y luego `sum(prices.values())`.

In [7]:
def calculate_total_price(customer_orders):
    total_price = sum(float(input(f"Enter the price of {product}: ")) for product in customer_orders)
    return total_price

### Paso 7. Programa principal

**Qué hace:** llama a las funciones en orden y pasa los resultados de una a otra.

**Por qué este orden:** las estadísticas se calculan con el pedido, así que pueden ir antes o después de actualizar el inventario. Sigo el orden del ejemplo del enunciado: estadísticas, inventario actualizado y, al final, el precio total.

**Nota:** los `input()` piden los datos al ejecutar. Para reproducir el ejemplo del enunciado: cantidades 5, 4, 3, 2, 1; 2 pedidos (`hat` y `keychain`); precios 10 y 5. Como el pedido es un `set`, el orden en que se piden los precios puede cambiar de una ejecución a otra; el total es el mismo.

In [8]:
inventory = initialize_inventory(products)
customer_orders = get_customer_orders()

order_statistics = calculate_order_statistics(customer_orders, products)
print_order_statistics(order_statistics)

inventory = update_inventory(customer_orders, inventory)
print_updated_inventory(inventory)

total_price = calculate_total_price(customer_orders)
print(f"Total Price: {total_price}")

Enter the quantity of t-shirts available: 5
Enter the quantity of mugs available: 4
Enter the quantity of hats available: 3
Enter the quantity of books available: 2
Enter the quantity of keychains available: 1
Enter the number of customer orders: 2
Enter the name of a product that a customer wants to order: hat
Enter the name of a product that a customer wants to order: keychain

Order Statistics:
Total Products Ordered: 2
Percentage of Unique Products Ordered: 40.0

Updated Inventory:
t-shirt: 5
mug: 4
hat: 2
book: 2
Enter the price of keychain: 10
Enter the price of hat: 5
Total Price: 15.0
